In [0]:
from pyspark.sql.functions import col
from pyspark.sql.functions import to_timestamp, col, date_trunc, hour, expr, regexp_replace, try_to_timestamp

In [0]:
TEAM_DIR = "dbfs:/student-groups/Group_3_2"
display(dbutils.fs.ls("dbfs:/mnt/mids-w261"))

In [0]:
display(dbutils.fs.ls("dbfs:/mnt/mids-w261/OTPW_60M/OTPW_60M"))

In [0]:
otpw_full = spark.read.parquet("dbfs:/mnt/mids-w261/OTPW_60M/OTPW_60M")
weather_full = spark.read.parquet('dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_weather_data/')
display(weather_full)

In [0]:
display(otpw_full)

In [0]:
def rejoin_otpw_and_weather(otpw_df, weather_df):
    '''Rejoin OTPW and Weather DF such that recorded weather observations are at least 2 hours before a flight's scheduled departure'''
    # filter on fm-15
    df_weather_filtered = weather_df.filter(col('REPORT_TYPE') == 'FM-15')   

    stations_set = {
        row["origin_station_id"]
        for row in otpw_df.select("origin_station_id").distinct().collect()
    }

    # filter on only stations in otpw
    df_weather_filtered = df_weather_filtered.filter(col('station').isin(stations_set))

    # print(f'Row Count (filter stationid): {df_weather_filtered.count()}')

    # Fix malformed timestamps first
    df_weather_filtered = df_weather_filtered.withColumn(
        "DATE",
        regexp_replace(col("DATE"), r'T(\d):', r'T0\1:')
    )

    df_weather_filtered = df_weather_filtered.withColumn(
        "DATE",
        regexp_replace(col("DATE"), r':(\d):', r':0\1:')
    )

    # Convert to timestamp. Create "ts" from "DATE"
    df_weather_filtered = df_weather_filtered.withColumn(
        "ts",
        to_timestamp(col("DATE"), "yyyy-MM-dd'T'HH:mm:ss")
    )

    df_weather_filtered = df_weather_filtered.filter(col("ts").isNotNull())

    # Truncate to hour. Create hour_trunc from "ts"
    df_weather_filtered = df_weather_filtered.withColumn(
        "hour_trunc",
        date_trunc("hour", col("ts"))
    )

    # List of columns to keep
    cols_to_keep = [
        'STATION',
        'DATE',
        'REPORT_TYPE',
        'SOURCE',
        'HourlyDryBulbTemperature',
        'HourlyPrecipitation',
        'HourlyRelativeHumidity',
        'HourlyVisibility',
        'HourlyWindSpeed',
        'hour_trunc'
    ]

    # Filter the DataFrame to only these columns
    df_weather_filtered = df_weather_filtered.select(*cols_to_keep)

    # In df_weather_filtered, rename STATION → origin_station_id to match OTPW
    df_weather_filtered = df_weather_filtered.withColumnRenamed("STATION", "origin_station_id")

    weather_cols = [
        'HourlyDryBulbTemperature',
        'HourlyPrecipitation',
        'HourlyRelativeHumidity',
        'HourlyVisibility',
        'HourlyWindSpeed'
    ]

    df_weather_renamed = df_weather_filtered
    for col_name in weather_cols:
        df_weather_renamed = df_weather_renamed.withColumnRenamed(col_name, f"2h_{col_name}")

    # drop duplicates
    df_weather_renamed = df_weather_renamed.dropDuplicates(["origin_station_id", "hour_trunc"])

    # Drop columns that are no longer needed (original hourly weather vars and DATE)
    df_weather_renamed = df_weather_renamed.drop(*weather_cols)
    df_weather_renamed = df_weather_renamed.drop('DATE')

    otpw_df = otpw_df.withColumn(
        "sched_depart_date_time",
        regexp_replace(col("sched_depart_date_time"), r'T(\d):', r'T0\1:')
    )

    otpw_df = otpw_df.withColumn(
        "sched_depart_date_time",
        regexp_replace(col("sched_depart_date_time"), r':(\d):', r':0\1:')
    )

    otpw_df = otpw_df.withColumn("ts", to_timestamp(col("sched_depart_date_time"), "yyyy-MM-dd'T'HH:mm:ss"))
    # Optional safety
    otpw_df = otpw_df.filter(col("ts").isNotNull())
    otpw_df = otpw_df.withColumn("hour_trunc", date_trunc("hour", col("ts") - expr("INTERVAL 2 HOURS")))

    otpw_df_2h = otpw_df.join(
        df_weather_renamed,
        on=["origin_station_id", "hour_trunc"],  # join keys
        how="left"
    )

    return otpw_df_2h

def get_dimensions(df):
    print(f"Rows: {df.count()}")
    print(f"Columns: {len(df.columns)}")

# Load 3 month data

In [0]:
# Weather Data 3months
df_weather_3m = spark.read.parquet(f"dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_weather_data_3m/")

# OTPW 3 months
otpw_3m = spark.read.parquet(f"{TEAM_DIR}/otpw_3m.parquet")

In [0]:

display(otpw_3m_2h)

In [0]:
otpw_3m_2h = rejoin_otpw_and_weather(otpw_3m, df_weather_3m)

get_dimensions(otpw_3m)
get_dimensions(otpw_3m_2h)

display(otpw_3m_2h)

In [0]:
# From weather table, Create "ts" from "DATE"
# From weather table, create "hour_trunc" from 'ts'. We do not keep 'ts' in weather anymore.

# From otpw table, Create "ts" from "sched_depart_date_time"
# From otpw table, create "hour_trunc" by subtracting 2 hours from "ts"

# The main hour_trunc in the joined dataset comes from OTPW. 
# We left join otpw and weather on [hour_trunc, station_id] so that the hour_trunc that shows up should be 2 hours before the departure time.
# hour_trunc 2 hours before scheduled departure time should also match with weather table's hour_trunc, which should give us weather data 2 hours before scheduled departure.

# The "DATE" column corresponds with the original Hourly Weather variables. Does not correspond with "sched_depart_date_time".
# The "2h_HourlyDryBulbTemperature" and other 2h variables should match the hour_trunc key from otpw.
display(otpw_3m_2h) # show hour_trunc, FL_DATE, CRS_DEP_TIME, sched_depart_date_time, DATE

In [0]:
# airlines.FL_DATE: Flight Date
# airlines.CRS_DEP_TIME: Computer Reservation System Departure Time (scheduled), in local time (ex. 1810, 2055, etc.)
# weather.DATE: The date and time the weather was recorded
# airlines.ORIGIN: Origin Airport code
# origin_iata_code
# weather.LATITUDE: 
# weather.LONGITUDE:
# stations.lat
# stations.lon

In [0]:
otpw_3m = spark.read.parquet(f"{TEAM_DIR}/otpw_3m.parquet")
display(otpw_3m)

In [0]:
display(otpw_3m.select(['FL_DATE', 'CRS_DEP_TIME', 'DATE']))

In [0]:
# origin_station_id (otpw) -> station (df_weather) 

# two_hours_prior_depart_UTC -> convert to nearest hour

# DATE -> convert to nearest hour

# df and df_weather

# 1. filter weather on only mf15 report type
# 2. 

In [0]:


print(f'Row Count: {df_weather_3m.count()}')

# filter on fm-15
df_weather_filtered = df_weather_3m.filter(col('REPORT_TYPE') == 'FM-15')

print(f'Row Count (filter fm15): {df_weather_filtered.count()}')
# get all unique sets in otpw
stations_set = {
    row["origin_station_id"]
    for row in otpw_3m.select("origin_station_id").distinct().collect()
}

# filter on only stations in otpw
df_weather_filtered = df_weather_filtered.filter(col('station').isin(stations_set))

print(f'Row Count (filter stationid): {df_weather_filtered.count()}')

#convert to nearest hour
df_weather_filtered = df_weather_filtered.withColumn("ts", to_timestamp(col("DATE"), "yyyy-MM-dd'T'HH:mm:ss"))
df_weather_filtered = df_weather_filtered.withColumn("hour_trunc", date_trunc("hour", col("ts")))



In [0]:
# List of columns to keep
cols_to_keep = [
    'STATION',
    'DATE',
    'REPORT_TYPE',
    'SOURCE',
    'HourlyAltimeterSetting',
    'HourlyDewPointTemperature',
    'HourlyDryBulbTemperature',
    'HourlyPrecipitation',
    'HourlyPresentWeatherType',
    'HourlyPressureChange',
    'HourlyPressureTendency',
    'HourlyRelativeHumidity',
    'HourlySkyConditions',
    'HourlySeaLevelPressure',
    'HourlyStationPressure',
    'HourlyVisibility',
    'HourlyWetBulbTemperature',
    'HourlyWindDirection',
    'HourlyWindGustSpeed',
    'HourlyWindSpeed',
    'ts',
    'hour_trunc'
]

# Filter the DataFrame to only these columns
df_weather_filtered = df_weather_filtered.select(*cols_to_keep)

In [0]:
from pyspark.sql.functions import to_timestamp, col, date_trunc, expr

#convert to nearest hour
otpw_3m = otpw_3m.withColumn("ts", to_timestamp(col("sched_depart_date_time"), "yyyy-MM-dd'T'HH:mm:ss"))
#otpw_3m = otpw_3m.withColumn("hour_trunc", date_trunc("hour", col("ts")))
otpw_3m = otpw_3m.withColumn("hour_trunc", date_trunc("hour", col("ts") - expr("INTERVAL 2 HOURS")))

display(otpw_3m)

In [0]:
# In df_weather_filtered, rename STATION → origin_station_id to match OTPW
df_weather_filtered = df_weather_filtered.withColumnRenamed("STATION", "origin_station_id")

weather_cols = [
    'HourlyAltimeterSetting',
    'HourlyDewPointTemperature',
    'HourlyDryBulbTemperature',
    'HourlyPrecipitation',
    'HourlyPresentWeatherType',
    'HourlyPressureChange',
    'HourlyPressureTendency',
    'HourlyRelativeHumidity',
    'HourlySkyConditions',
    'HourlySeaLevelPressure',
    'HourlyStationPressure',
    'HourlyVisibility',
    'HourlyWetBulbTemperature',
    'HourlyWindDirection',
    'HourlyWindGustSpeed',
    'HourlyWindSpeed'
]

df_weather_renamed = df_weather_filtered
for col_name in weather_cols:
    df_weather_renamed = df_weather_renamed.withColumnRenamed(col_name, f"2h_{col_name}")

# drop duplicates
df_weather_renamed = df_weather_renamed.dropDuplicates(["origin_station_id", "hour_trunc"])

otpw_3m_2h = otpw_3m.join(
    df_weather_renamed,
    on=["origin_station_id", "hour_trunc"],  # join keys
    how="left"
)

In [0]:
display(otpw_3m.filter(otpw_3m["origin_station_id"] == '72219013874'))


In [0]:

# Convert DATE to timestamp first
df_weather_3m = df_weather_3m.withColumn(
    "DATE_ts",
    to_timestamp(col("DATE"), "yyyy-MM-dd'T'HH:mm:ss")
)

# Filter by station and date range
display(
    df_weather_3m.filter(col("STATION") == '72219013874')
                 .filter(col("REPORT_TYPE") == 'FM-15')
                 .filter(
                     col("DATE_ts").between(
                         "2015-01-14 22:00:00",
                         "2015-01-14 23:00:00"
                     )
                 )
)

In [0]:
display(otpw_3m_2h)